In [4]:
import pandas as pd
import matplotlib as plt
from pyvis import network as net
import networkx as nx

# toon alle columns
pd.set_option('display.max_columns', None)

In [ ]:
ai_2020 = pd.read_excel("inputs/TrajectenAI2020.xlsx")
tot = len(ai_2020['Student'].drop_duplicates())
ai_2020.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 287 entries, 0 to 286
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Student                     287 non-null    int64 
 1   Opleiding ID                287 non-null    int64 
 2   Opleiding Omschrijving      287 non-null    object
 3   Startjaar(4)                287 non-null    int64 
 4   Startjaar                   287 non-null    object
 5   Eindjaar                    287 non-null    object
 6   Doorloop: Diploma behaald?  287 non-null    object
 7   Aantal studenten            287 non-null    int64 
dtypes: int64(4), object(4)
memory usage: 18.1+ KB


In [5]:
# opl_counts = ai_2020['Student'].value_counts()
met_traject = ai_2020[ai_2020.duplicated(subset=['Student'], keep=False)]

0      51016880
1      51016880
2      51016880
3      51016872
4      51016880
         ...   
282    51016880
283    51016880
284    51016880
285    51016880
286    51016880
Name: Opleiding ID, Length: 287, dtype: int64

In [32]:
s = len(met_traject['Student'].drop_duplicates())
o = len(met_traject['Opleiding ID'].drop_duplicates())
print(f"Unieke studenten: {s}, unieke opleidingen: {o}")

Unieke studenten: 73, unieke opleidingen: 69


In [ ]:
# Met juiste volgorde
opl_gr = met_traject.groupby(['Opleiding ID', 'Opleiding Omschrijving']).count()
opl_d = opl_gr.to_dict()['Student']
nodes, values, labels, colors = [], [], [], []
for key in opl_d:
    nodes.append(key[0])
    labels.append(key[1])
    values.append(opl_d[key])

nodes_str = [str(x) for x in nodes]

# kleurekes
for v in values:
    if v < 2:
        colors.append('#f19167')
    elif v < 5:
        colors.append('#51cb9e')
    elif v < 11:
        colors.append('#5f97d7')
    else:
        colors.append('#f3df5f')

gr=net.Network(notebook=True, cdn_resources='in_line')
gr.add_nodes(nodes, value=values, label=labels, title=nodes_str, color=colors)

# Adding edges
from collections import defaultdict

trajecten_by_student = met_traject[["Student", "Opleiding ID"]].groupby("Student")
t_b_s_dict = {student: opleiding["Opleiding ID"].to_list() for student, opleiding in trajecten_by_student}
#t_b_s_dict

def return_zero():
    return(0)

edges = defaultdict(return_zero)
for opleidingen in t_b_s_dict.values():
    #print(opleidingen)
    if (num_opleidingen := len(opleidingen)) > 1:
        for o_index in range(0, num_opleidingen-1):
            edges[(opleidingen[o_index], opleidingen[o_index+1])] += 1
            
for edge in edges:
    gr.add_edge(edge[0], edge[1], value=edges[edge])
    
#for student, opleidingen in trajecten_by_student:
    #print(opleidingen["Opleiding"].to_list)
gr.show("outputs/MNM_AI2020.html")

MNM_AI2020.html
